# Relatórios de Validação — Segmentação de LGD

Notebook **só com os relatórios de validação** da régua de LGD construída com
`yggdrasil.credit_risk.lgd.SequentialLGDSegmenter` — sob CMN 4.966/2021 e IFRS 9.

Cobre: **folhas/régua**, **PSI**, **CSI por variável**, **métricas**, **IC bootstrap**,
**backtest por safra**, **monotonicidade**, **calibração**, **qualidade dos segmentos**
e o **relatório consolidado** (Markdown).

> **Como usar com seus dados:** substitua a célula de *dados sintéticos* por uma que
> carregue o seu DataFrame, mantendo o contrato: coluna-alvo `lgd`, coluna de amostra
> `amostra` (com `DES` como referência) e coluna de safra `dt_ref`.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from yggdrasil.credit_risk.lgd import SequentialLGDSegmenter

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

## 2. Dados

_Dados sintéticos só para o notebook rodar de ponta a ponta — **troque pela sua base**._

In [ ]:
rng = np.random.default_rng(42)
n = 12000
safra = rng.choice(['2022Q1','2022Q2','2022Q3','2022Q4','2023Q1','2023Q2'], n)
amostra = np.where(np.isin(safra, ['2023Q1','2023Q2']), 'OOT', 'DES')
ltv = rng.uniform(0.3, 1.3, n)
ltv[rng.random(n) < 0.08] = np.nan                      # faltantes
garantia = rng.choice(['Imovel','Veiculo','Aval','Recebiveis'], n, p=[.4,.3,.2,.1]).astype(object)
_lg = {'Imovel':0.10, 'Veiculo':0.25, 'Aval':0.40, 'Recebiveis':0.55}
base = (0.1 + 0.35*np.nan_to_num(ltv-0.6, nan=0.35)
        + np.array([_lg[g] for g in garantia]) + rng.normal(0, 0.06, n))
df = pd.DataFrame({'ltv': ltv, 'garantia': garantia, 'dt_ref': safra,
                   'lgd': np.clip(base, 0, 1), 'amostra': amostra})
df.head()

## 3. Régua a validar

Construção rápida (árvore gulosa + poda) só para ter o que validar.

In [ ]:
seg = SequentialLGDSegmenter(df, target='lgd', sample_col='amostra', ref_sample='DES',
                             feature_labels={'ltv':'LTV', 'garantia':'Garantia'})
seg.fit_auto(max_depth=3)
seg.prune(min_repr=3.0, min_lgd_gap=0.03)               # funde irmãs fracas/próximas
_ = seg.tree()

## 4. Folhas / régua (com PSI e teste de hipótese entre irmãs)

In [ ]:
seg.leaves(with_psi=True, with_test=True)

## 5. PSI — estabilidade da segmentação (DES → demais amostras)

`<0.10` estável · `0.10–0.25` atenção · `≥0.25` instável.

In [ ]:
display(seg.psi())
seg.psi_detalhe().head(20)                              # contribuição por folha

## 6. CSI por variável — estabilidade das entradas

PSI aplicado à distribuição de cada variável de entrada (bins fixados no DES).

In [ ]:
display(seg.csi())
seg.csi_detalhe().head(20)

## 7. Métricas — régua como modelo de LGD

Prediz o LGD pela média do segmento no DES; avaliada como modelo em cada amostra.

In [ ]:
seg.metrics()

## 8. Intervalo de confiança (bootstrap) e aderência OOT

IC da média de LGD por folha via bootstrap no DES; verifica se o LGD de OOT cai dentro.

In [ ]:
seg.bootstrap_ci(n_boot=1000)

## 9. Backtest por safra — previsto × realizado no tempo

In [ ]:
seg.backtest('dt_ref')

## 10. Monotonicidade do LGD nas notas

Por construção o DES é monotônico; o valor está em conferir o OOT (estabilidade do rank).

In [ ]:
seg.monotonicity_report()

## 11. Calibração — previsto (DES) × realizado (OOT) por folha

In [ ]:
display(seg.calibration_table())
fig = seg.plot_calibration(); plt.show()

## 12. Qualidade dos segmentos

In [ ]:
fig = seg.plot_leaf_boxplots(); plt.show()           # dispersão do LGD por folha
fig = seg.plot_target_hist(by_sample=True); plt.show()  # distribuição do LGD

## 13. Árvore (imagem)

In [ ]:
fig = seg.plot_tree(); plt.show()

## 14. Relatório consolidado (Markdown)

Reúne tudo num único arquivo `.md` (com as imagens ao lado).

In [ ]:
caminho = seg.validation_report('relatorio_validacao_lgd.md', time_col='dt_ref')
print('Relatório gerado em:', caminho)